# SDK数据探索分析

本notebook用于探索和分析SDK样本数据，了解数据分布和特征覆盖情况。

In [ ]:
import sys
sys.path.append('/Users/chengqihan/AICode/sdk_clustering/src')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

sns.set_style("whitegrid")
sns.set_palette("husl")

In [ ]:
from feature_engineer import SDKFeatureEngineer

engineer = SDKFeatureEngineer()

## 1. 加载数据

In [ ]:
data_path = '/Users/chengqihan/AICode/sdk_clustering/data/raw/samples.json'

try:
    data = engineer.load_data(data_path)
    print(f"成功加载 {len(data)} 个样本")
except FileNotFoundError:
    print(f"数据文件未找到: {data_path}")
    print("请将你的JSON数据文件放到指定位置")

## 2. 数据基本信息

In [ ]:
if data:
    print(f"总样本数: {len(data)}")
    print(f"\n样本示例:")
    print(json.dumps(data[0], indent=2, ensure_ascii=False)[:500] + "...")

## 3. 特征提取分析

In [ ]:
if data:
    sample_infos = [engineer.extract_features_per_sample(sample) for sample in data]
    
    df = pd.DataFrame([{
        'coordinateName': info['coordinateName'],
        'version': info['version'],
        'import_export_count': len(info['import_export_sequences']),
        'opcode_count': len(info['opcode_sequences']),
        'tlsh_count': len(info['tlsh_hashes']),
        'total_sequences': info['total_sequences']
    } for info in sample_infos])
    
    display(df.head(10))

## 4. 特征覆盖率统计

In [ ]:
if data:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    axes[0, 0].pie([
        len(df[df['import_export_count'] > 0]),
        len(df[df['import_export_count'] == 0])
    ], labels=['有导入导出序列', '无导入导出序列'], autopct='%1.1f%%')
    axes[0, 0].set_title('导入导出序列覆盖率')
    
    axes[0, 1].pie([
        len(df[df['opcode_count'] > 0]),
        len(df[df['opcode_count'] == 0])
    ], labels=['有操作码序列', '无操作码序列'], autopct='%1.1f%%')
    axes[0, 1].set_title('操作码序列覆盖率')
    
    axes[1, 0].pie([
        len(df[df['tlsh_count'] > 0]),
        len(df[df['tlsh_count'] == 0])
    ], labels=['有TLSH哈希', '无TLSH哈希'], autopct='%1.1f%%')
    axes[1, 0].set_title('TLSH哈希覆盖率')
    
    
    seq_counts = df['total_sequences'].value_counts().sort_index()
    axes[1, 1].bar(seq_counts.index, seq_counts.values)
    axes[1, 1].set_xlabel('序列数量')
    axes[1, 1].set_ylabel('样本数')
    axes[1, 1].set_title('序列数量分布')
    
    plt.tight_layout()
    plt.show()

## 5. 序列长度分布

In [ ]:
if data:
    import_export_lengths = []
    opcode_lengths = []
    
    for info in sample_infos:
        for seq in info['import_export_sequences']:
            import_export_lengths.append(len(seq.split()))
        for seq in info['opcode_sequences']:
            opcode_lengths.append(len(seq.split()))
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    if import_export_lengths:
        axes[0].hist(import_export_lengths, bins=30, alpha=0.7, edgecolor='black')
        axes[0].set_xlabel('序列长度（token数）')
        axes[0].set_ylabel('频次')
        axes[0].set_title('导入导出序列长度分布')
    
    if opcode_lengths:
        axes[1].hist(opcode_lengths, bins=30, alpha=0.7, edgecolor='black', color='orange')
        axes[1].set_xlabel('序列长度（token数）')
        axes[1].set_ylabel('频次')
        axes[1].set_title('操作码序列长度分布')
    
    plt.tight_layout()
    plt.show()

## 6. 统计摘要

In [ ]:
if data:
    print("="*50)
    print("数据探索分析报告")
    print("="*50)
    print(f"\n总样本数: {len(df)}")
    print(f"\n特征覆盖率:")
    print(f"  - 有导入导出序列: {len(df[df['import_export_count'] > 0])} ({len(df[df['import_export_count'] > 0])/len(df)*100:.1f}%)")
    print(f"  - 有操作码序列: {len(df[df['opcode_count'] > 0])} ({len(df[df['opcode_count'] > 0])/len(df)*100:.1f}%)")
    print(f"  - 有TLSH哈希: {len(df[df['tlsh_count'] > 0])} ({len(df[df['tlsh_count'] > 0])/len(df)*100:.1f}%)")
    print(f"\n序列统计:")
    print(f"  - 平均序列数: {df['total_sequences'].mean():.1f}")
    print(f"  - 最大序列数: {df['total_sequences'].max()}")
    print(f"  - 最小序列数: {df['total_sequences'].min()}")
    print(f"  - 中位数: {df['total_sequences'].median():.1f}")